# AorticTurb — canonical Pass 3 execution

This notebook is a **thin execution driver** for the public `khalid-saqr/AorticTurb` package. It does not reimplement turbulence calculations.

Canonical scientific scope for this run:

1. mandatory startup/integrity gate;
2. fresh phase-conditioned decomposition for M/AAo at `t018`, `t021`, `t024`, with 16 independent phase-locked realizations per phase;
3. Gate A using the unchanged primary implementation;
4. the pre-specified support-policy sensitivity analysis.

The notebook does **not** call `run-all`, KO, Taylor, Lumley, canonical diagnostics, integrated classification, or promote anything into root `05_results`.

Scientific package commit pinned for this run: `4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5`.


In [27]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Always move to a stable directory before deleting/reinstalling anything.
os.chdir("/content")

AORTICTURB_COMMIT = "43544057e627f0a00051557171fafdc2e41b7402"
EXPECTED_AORTICTURB_VERSION = "0.2.0"

# Build the public Git URL programmatically so no notebook/chat renderer can
# convert it into Markdown-link syntax.
REPO_HOST = "github" + ".com"
REPO_PATH = "khalid-saqr/" + "AorticTurb.git"
REPO_URL = "https://" + REPO_HOST + "/" + REPO_PATH
AORTICTURB_SPEC = "git+" + REPO_URL + "@" + AORTICTURB_COMMIT

# Install the pinned package in an isolated directory without using venv.
SITE_DIR = Path("/content/aorticturb-pass3-site")
BIN_DIR = Path("/content/aorticturb-pass3-bin")

for path in (SITE_DIR, BIN_DIR):
    if path.exists():
        shutil.rmtree(path)

SITE_DIR.mkdir(parents=True, exist_ok=True)
BIN_DIR.mkdir(parents=True, exist_ok=True)

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-deps",
        "--target",
        str(SITE_DIR),
        AORTICTURB_SPEC,
    ],
    cwd="/content",
    capture_output=True,
    text=True,
)

if install.returncode != 0:
    raise RuntimeError(
        "AORTICTURB_INSTALL_FAILED\n"
        + (install.stderr.strip() or install.stdout.strip())
    )

# Verify pip recorded the exact requested Git commit.
dist_infos = list(SITE_DIR.glob("aorticturb-*.dist-info"))
if len(dist_infos) != 1:
    raise RuntimeError(
        f"AORTICTURB_INSTALL_FAILED: expected one dist-info directory, found {dist_infos}"
    )

direct_url_path = dist_infos[0] / "direct_url.json"
if not direct_url_path.is_file():
    raise RuntimeError(
        f"AORTICTURB_PROVENANCE_FAILED: missing {direct_url_path}"
    )

direct_url = json.loads(direct_url_path.read_text(encoding="utf-8"))
INSTALLED_COMMIT = direct_url.get("vcs_info", {}).get("commit_id")

if INSTALLED_COMMIT != AORTICTURB_COMMIT:
    raise RuntimeError(
        f"AORTICTURB_COMMIT_MISMATCH: expected {AORTICTURB_COMMIT}, "
        f"got {INSTALLED_COMMIT}"
    )

# Every scientific command will run in a fresh interpreter with this pinned
# package first on PYTHONPATH. The notebook kernel never imports aorticturb.
VENV_PYTHON = Path(sys.executable)
AORTICTURB = BIN_DIR / "aorticturb"

wrapper = f"""#!/bin/sh
export PYTHONPATH="{SITE_DIR}${{PYTHONPATH:+:$PYTHONPATH}}"
exec "{sys.executable}" -c 'from aorticturb.cli import app; app()' "$@"
"""

AORTICTURB.write_text(wrapper, encoding="utf-8")
AORTICTURB.chmod(0o755)

# Verify package version and import location in a fresh subprocess.
env = os.environ.copy()
env["PYTHONPATH"] = str(SITE_DIR) + (
    os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import aorticturb; "
            "print(aorticturb.__version__); "
            "print(aorticturb.__file__)"
        ),
    ],
    cwd="/content",
    env=env,
    capture_output=True,
    text=True,
)

if probe.returncode != 0:
    raise RuntimeError(
        "AORTICTURB_IMPORT_FAILED\n"
        + (probe.stderr.strip() or probe.stdout.strip())
    )

probe_lines = probe.stdout.strip().splitlines()
installed_version = probe_lines[0]
loaded_from = Path(probe_lines[1]).resolve()

if installed_version != EXPECTED_AORTICTURB_VERSION:
    raise RuntimeError(
        f"AORTICTURB_VERSION_MISMATCH: expected {EXPECTED_AORTICTURB_VERSION}, "
        f"got {installed_version}"
    )

if SITE_DIR.resolve() not in loaded_from.parents:
    raise RuntimeError(
        f"AORTICTURB_IMPORT_PROVENANCE_FAILED: loaded from {loaded_from}, "
        f"expected under {SITE_DIR}"
    )

print("AorticTurb:", installed_version)
print("Pinned scientific commit:", INSTALLED_COMMIT)
print("Loaded from:", loaded_from)
print("CLI:", AORTICTURB)

AorticTurb: 0.2.0
Pinned scientific commit: 43544057e627f0a00051557171fafdc2e41b7402
Loaded from: /content/aorticturb-pass3-site/aorticturb/__init__.py
CLI: /content/aorticturb-pass3-bin/aorticturb


In [28]:
import hashlib
import os
import shutil
from pathlib import Path
from google.colab import drive

# Mount only if Drive is not already available.
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study"
).resolve()

WORKSPACE_ROOT = PROJECT_ROOT / "results"
ANALYSIS_ROOT = WORKSPACE_ROOT / "04_analysis" / "01_laminar_vs_unsteady"

# AorticTurb's verifier expects a package-style SHA256SUMS.txt at DATASET_ROOT.
# The canonical project intentionally stores its authoritative checksum ledger
# under 00_provenance/checksums/, so build a lightweight scratch VIEW of the
# canonical dataset using symlinks. No 2.2-GB data are copied or modified.
DATASET_ROOT = Path("/content/aorticturb-pass3-input")

PHASES = ("t018", "t021", "t024")
REALIZATIONS = tuple(f"B{i:05d}.npy" for i in range(1, 17))

MANIFEST_REL_PATHS = (
    Path("00_provenance/transfer_manifests/pass5_integrity_audit.json"),
    Path("00_provenance/transfer_manifests/acquisition_manifest_v1.json"),
    Path("00_provenance/transfer_manifests/geometry_manifest.json"),
)

HISTORICAL_LEDGER_REL = Path(
    "00_provenance/checksums/final_acquired_SHA256SUMS.txt"
)

GEOMETRY_REL_PATHS = (
    Path("02_metadata_geometry/M_medium_root/original_geometry/M_aortic_root.STL"),
    Path("02_metadata_geometry/S_small_root/original_geometry/S_aortic_root.STL"),
    Path("02_metadata_geometry/L_large_root/original_geometry/L_aortic_root.STL"),
    Path("02_metadata_geometry/NS_non_sinus_root/original_geometry/NS_aortic_root.STL"),
)

# -------------------------------------------------------------------------
# 1. Validate the actual canonical project files required by AGENTS.md.
# -------------------------------------------------------------------------
required_project_files = [
    PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "METADATA.json",
    PROJECT_ROOT / HISTORICAL_LEDGER_REL,
    *(PROJECT_ROOT / rel for rel in MANIFEST_REL_PATHS),
    *(PROJECT_ROOT / rel for rel in GEOMETRY_REL_PATHS),
]

for phase in PHASES:
    phase_dir = PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / phase
    required_project_files.extend(
        phase_dir / name for name in REALIZATIONS
    )

missing = [str(path) for path in required_project_files if not path.is_file()]
if missing:
    raise RuntimeError(
        "Canonical project is missing required files:\n" + "\n".join(missing)
    )

# Confirm exactly the canonical 16 realizations exist at each selected phase.
for phase in PHASES:
    phase_dir = PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / phase
    observed = sorted(path.name for path in phase_dir.glob("*.npy"))
    if observed != list(REALIZATIONS):
        raise RuntimeError(
            f"{phase}: expected exactly B00001.npy-B00016.npy; got {observed}"
        )

# -------------------------------------------------------------------------
# 2. Parse the authoritative historical checksum ledger.
# -------------------------------------------------------------------------
def parse_sha256_ledger(path):
    records = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            raise RuntimeError(f"Malformed checksum entry: {line}")

        digest, relpath = parts
        records[relpath.strip().lstrip("*")] = digest.lower()

    return records


historical_ledger_path = PROJECT_ROOT / HISTORICAL_LEDGER_REL
historical = parse_sha256_ledger(historical_ledger_path)

# Geometry entries in the historical ledger may use acquisition-era paths,
# so resolve their authoritative hashes uniquely by basename.
historical_by_basename = {}
for relpath, digest in historical.items():
    historical_by_basename.setdefault(Path(relpath).name, set()).add(digest)

# -------------------------------------------------------------------------
# 3. Create a scratch dataset VIEW using symlinks only.
# -------------------------------------------------------------------------
os.chdir("/content")

if DATASET_ROOT.exists() or DATASET_ROOT.is_symlink():
    if DATASET_ROOT.is_symlink() or DATASET_ROOT.is_file():
        DATASET_ROOT.unlink()
    else:
        shutil.rmtree(DATASET_ROOT)

DATASET_ROOT.mkdir(parents=True)


def link_file(source, destination):
    source = Path(source).resolve()
    destination = Path(destination)

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() or destination.is_symlink():
        destination.unlink()

    destination.symlink_to(source)


# Root project metadata.
link_file(PROJECT_ROOT / "AGENTS.md", DATASET_ROOT / "AGENTS.md")
link_file(PROJECT_ROOT / "METADATA.json", DATASET_ROOT / "METADATA.json")

# Required provenance.
for rel in MANIFEST_REL_PATHS:
    link_file(PROJECT_ROOT / rel, DATASET_ROOT / rel)

link_file(
    PROJECT_ROOT / HISTORICAL_LEDGER_REL,
    DATASET_ROOT / HISTORICAL_LEDGER_REL,
)

# Original geometry.
for rel in GEOMETRY_REL_PATHS:
    link_file(PROJECT_ROOT / rel, DATASET_ROOT / rel)

# Canonical M/AAo velocity subset only.
velocity_rel_paths = []

for phase in PHASES:
    for name in REALIZATIONS:
        rel = (
            Path("01_raw_data")
            / "M_medium_root"
            / "AAo"
            / phase
            / name
        )

        source = PROJECT_ROOT / rel

        if source.stat().st_size != 47_164_880:
            raise RuntimeError(
                f"Unexpected raw velocity size for {source}: "
                f"{source.stat().st_size} bytes"
            )

        link_file(source, DATASET_ROOT / rel)
        velocity_rel_paths.append(rel)

# -------------------------------------------------------------------------
# 4. Generate the package-level SHA256SUMS.txt required by AorticTurb.
#
# Velocity/geometry hashes come from the authoritative acquisition ledger.
# Only the small metadata/provenance files are hashed directly here.
# -------------------------------------------------------------------------
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()


checksum_records = {}

# Velocity hashes: exact canonical paths must already exist in the
# authoritative final-acquired ledger.
for rel in velocity_rel_paths:
    key = rel.as_posix()

    if key not in historical:
        raise RuntimeError(
            f"Authoritative checksum ledger does not contain {key}"
        )

    checksum_records[key] = historical[key]

# Geometry hashes: resolve uniquely by filename.
for rel in GEOMETRY_REL_PATHS:
    candidates = historical_by_basename.get(rel.name, set())

    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve authoritative checksum for {rel.name}: "
            f"{sorted(candidates)}"
        )

    checksum_records[rel.as_posix()] = next(iter(candidates))

# Small project/provenance files.
small_required = [
    Path("AGENTS.md"),
    Path("METADATA.json"),
    *MANIFEST_REL_PATHS,
    HISTORICAL_LEDGER_REL,
]

for rel in small_required:
    checksum_records[rel.as_posix()] = sha256_file(DATASET_ROOT / rel)

package_ledger = DATASET_ROOT / "SHA256SUMS.txt"

package_ledger.write_text(
    "".join(
        f"{checksum_records[rel]}  {rel}\n"
        for rel in sorted(checksum_records)
    ),
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# 5. Prepare analysis output only. Raw Drive files remain untouched.
# -------------------------------------------------------------------------
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

print("Canonical project root:", PROJECT_ROOT)
print("Analysis dataset view:", DATASET_ROOT)
print("Velocity fields linked:", len(velocity_rel_paths))
print("Package checksum entries:", len(checksum_records))
print("Analysis workspace:", WORKSPACE_ROOT)
print("Canonical root 05_results untouched:", PROJECT_ROOT / "05_results")

Canonical project root: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study
Analysis dataset view: /content/aorticturb-pass3-input
Velocity fields linked: 48
Package checksum entries: 58
Analysis workspace: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Canonical root 05_results untouched: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/05_results


In [29]:
def run_cli(*args):
    command = [str(AORTICTURB), *map(str, args)]
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"AorticTurb command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed

def runs_with(marker):
    if not ANALYSIS_ROOT.exists():
        return set()
    return {
        path.resolve()
        for path in ANALYSIS_ROOT.iterdir()
        if path.is_dir() and (path / marker).is_file()
    }

def run_stage(marker, *args):
    before = runs_with(marker)
    run_cli(*args)
    after = runs_with(marker)
    created = sorted(after - before)
    if len(created) != 1:
        raise RuntimeError(
            f"Expected exactly one new run containing {marker}; found {len(created)}: {created}"
        )
    return created[0]


## 1. Mandatory startup gate

This runs the package startup gate before any new scientific analysis. A failure stops the notebook.


In [30]:
run_cli("gate", "--root", DATASET_ROOT)
print("Startup gate: PASS")


$ /content/aorticturb-pass3-bin/aorticturb gate --root /content/aorticturb-pass3-input
Startup gate: PASS


## 2. Fresh phase-conditioned decomposition

No phases are pooled. Source NaNs remain measurement-support masks; raw data are not modified.


In [31]:
DECOMPOSITION_RUN = run_stage(
    "decomposition_summary.json",
    "decompose",
    "--root", DATASET_ROOT,
    "--output", WORKSPACE_ROOT,
)
print("Decomposition run:", DECOMPOSITION_RUN)


$ /content/aorticturb-pass3-bin/aorticturb decompose --root /content/aorticturb-pass3-input --output /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Decomposition run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_105838_983261


## 3. Gate A — turbulence versus deterministic unsteadiness

This uses the package's existing Gate-A implementation and its unchanged primary support rule.


In [32]:
import json
import os
import signal
import subprocess
from pathlib import Path

# -------------------------------------------------------------------------
# Gate A — turbulence versus deterministic unsteadiness
#
# Run the package driver directly in a fresh Python process.
# This avoids the notebook CLI wrapper and guarantees that stdout/stderr,
# Python tracebacks, and OS-level process termination are visible.
# -------------------------------------------------------------------------

if not Path(DECOMPOSITION_RUN).is_dir():
    raise RuntimeError(
        f"Decomposition run does not exist: {DECOMPOSITION_RUN}"
    )

if not (Path(DECOMPOSITION_RUN) / "decomposition_summary.json").is_file():
    raise RuntimeError(
        f"Missing decomposition_summary.json in {DECOMPOSITION_RUN}"
    )

if not (Path(DECOMPOSITION_RUN) / "decomposition_index.json").is_file():
    raise RuntimeError(
        f"Missing decomposition_index.json in {DECOMPOSITION_RUN}"
    )

# Confirm the processed arrays referenced by the decomposition actually exist
# before starting the expensive Gate-A computation.
decomp_index = json.loads(
    (Path(DECOMPOSITION_RUN) / "decomposition_index.json").read_text(
        encoding="utf-8"
    )
)

for phase in ("t018", "t021", "t024"):
    if phase not in decomp_index:
        raise RuntimeError(
            f"Decomposition index is missing phase {phase}"
        )

    array_path = (
        Path(DECOMPOSITION_RUN) / decomp_index[phase]
    ).resolve()

    if not array_path.is_file():
        raise RuntimeError(
            f"Processed decomposition array is missing for {phase}: "
            f"{array_path}"
        )

print("Using decomposition:", DECOMPOSITION_RUN)
print("Dataset:", DATASET_ROOT)
print("Workspace:", WORKSPACE_ROOT)

# Run Gate A directly through the scientific package API.
gate_a_script = r'''
import json
import os
import sys
import traceback
from pathlib import Path

try:
    from aorticturb.config import load_config
    from aorticturb.regime.driver import run_regime

    dataset_root = Path(os.environ["AT_DATASET_ROOT"])
    decomposition_run = Path(os.environ["AT_DECOMPOSITION_RUN"])
    workspace_root = Path(os.environ["AT_WORKSPACE_ROOT"])

    config = load_config(None)

    result = run_regime(
        dataset_root,
        decomposition_run,
        workspace_root,
        config,
    )

    print(
        "__AORTICTURB_GATE_A_RESULT__="
        + json.dumps(result),
        flush=True,
    )

except BaseException:
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise
'''

env = os.environ.copy()

# Ensure this fresh process imports only the pinned package installed in Cell 1.
env["PYTHONPATH"] = str(SITE_DIR) + (
    os.pathsep + env["PYTHONPATH"]
    if env.get("PYTHONPATH")
    else ""
)

env["AT_DATASET_ROOT"] = str(Path(DATASET_ROOT).resolve())
env["AT_DECOMPOSITION_RUN"] = str(Path(DECOMPOSITION_RUN).resolve())
env["AT_WORKSPACE_ROOT"] = str(Path(WORKSPACE_ROOT).resolve())

# Unbuffer Python so progress/errors appear immediately.
env["PYTHONUNBUFFERED"] = "1"

# Prevent numerical libraries from spawning unnecessary thread pools.
# This does not change the scientific definitions or Gate-A criteria.
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"

print("\nStarting Gate A...\n")

process = subprocess.Popen(
    [str(VENV_PYTHON), "-c", gate_a_script],
    cwd="/content",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

return_code = process.wait()

# -------------------------------------------------------------------------
# Give an explicit diagnosis even when the operating system kills Python
# without allowing it to emit a traceback.
# -------------------------------------------------------------------------
if return_code != 0:
    if return_code == -signal.SIGKILL or return_code == 137:
        raise RuntimeError(
            "GATE_A_PROCESS_KILLED: the Gate-A Python process was killed "
            "by SIGKILL. In Colab this normally indicates an out-of-memory "
            "or runtime resource kill. No Python traceback can be emitted "
            "after SIGKILL."
        )

    if return_code == -signal.SIGSEGV or return_code == 139:
        raise RuntimeError(
            "GATE_A_PROCESS_CRASHED: the Gate-A process terminated with "
            "SIGSEGV. See the streamed output above."
        )

    raise RuntimeError(
        f"GATE_A_FAILED: process exited with code {return_code}. "
        "The complete process output is shown above."
    )

# -------------------------------------------------------------------------
# Recover the exact run directory returned by run_regime().
# -------------------------------------------------------------------------
result_prefix = "__AORTICTURB_GATE_A_RESULT__="

result_lines = [
    line.strip()
    for line in output_lines
    if line.startswith(result_prefix)
]

if len(result_lines) != 1:
    raise RuntimeError(
        "Gate A exited successfully but did not return exactly one "
        "result record."
    )

regime_result = json.loads(
    result_lines[0][len(result_prefix):]
)

REGIME_RUN = Path(regime_result["run_dir"]).resolve()

if not REGIME_RUN.is_dir():
    raise RuntimeError(
        f"Gate-A run directory was not created: {REGIME_RUN}"
    )

required_outputs = (
    "run_manifest.json",
    "regime_summary.json",
    "metrics.csv",
    "diagnostics.json",
)

missing_outputs = [
    name
    for name in required_outputs
    if not (REGIME_RUN / name).is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Gate A returned successfully but required outputs are missing: "
        + ", ".join(missing_outputs)
    )

manifest = json.loads(
    (REGIME_RUN / "run_manifest.json").read_text(encoding="utf-8")
)

if manifest.get("status") != "COMPLETE":
    raise RuntimeError(
        f"Gate-A manifest status is {manifest.get('status')!r}, "
        "expected 'COMPLETE'."
    )

print("\nGate A completed successfully.")
print("Gate-A run:", REGIME_RUN)
print("Recorded commit:", manifest.get("git_commit"))

Using decomposition: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_105838_983261
Dataset: /content/aorticturb-pass3-input
Workspace: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results

Starting Gate A...

/content/aorticturb-pass3-site/aorticturb/regime/gate_a.py:34: RuntimeWarning: Mean of empty slice
  rms = np.sqrt(np.nanmean(np.sum(residuals**2, axis=-1), axis=0))
__AORTICTURB_GATE_A_RESULT__={"run_dir": "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_105959_784433", "phases": {"t018": {"decision": "UNRESOLVED", "turbulence_established": null}, "t021": {"decision": "STOCHASTIC_MULTISCALE_SUPPORTED", "turbulence_established": true}, "t024": {"decision": "STOCHASTIC_MULTISCALE_SUPPORTED", "turbulence_established": true}}}

Gate A completed successfully.
Gate-A run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Stud

## 4. Pre-specified support-policy sensitivity analysis

This evaluates the locked support-policy comparators against the fresh decomposition and Gate-A evidence. It does not replace the primary Gate-A result.


In [33]:
SENSITIVITY_RUN = run_stage(
    "sensitivity_summary.json",
    "support-sensitivity",
    "--root", DATASET_ROOT,
    "--decomposition", DECOMPOSITION_RUN,
    "--regime", REGIME_RUN,
    "--output", WORKSPACE_ROOT,
)
print("Support-sensitivity run:", SENSITIVITY_RUN)


$ /content/aorticturb-pass3-bin/aorticturb support-sensitivity --root /content/aorticturb-pass3-input --decomposition /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_105838_983261 --regime /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_105959_784433 --output /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Support-sensitivity run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_110036_139250_support_sensitivity


## 5. Validate and display the fresh evidence

The checks below are execution/provenance checks, not new turbulence criteria.


In [34]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

PHASES = {"t018", "t021", "t024"}
POLICIES = {
    "legacy_global_fraction",
    "strict_mask_consistency",
    "fully_finite_local_windows",
    "strict_mask_and_local_windows",
}

decomposition_metrics = pd.read_csv(DECOMPOSITION_RUN / "metrics.csv")
required_support_columns = {
    "phase",
    "support_fraction",
    "common_support_nodes",
    "union_support_nodes",
    "intersection_over_union_fraction",
    "masks_identical",
    "realizations",
}
missing_columns = required_support_columns - set(decomposition_metrics.columns)
if missing_columns:
    raise RuntimeError(f"Fresh decomposition is missing required support columns: {sorted(missing_columns)}")
if set(decomposition_metrics["phase"]) != PHASES:
    raise RuntimeError("Fresh decomposition does not contain exactly t018, t021, t024")
if set(decomposition_metrics["realizations"].astype(int)) != {16}:
    raise RuntimeError("Fresh decomposition does not report 16 realizations per phase")

regime_summary = json.loads((REGIME_RUN / "regime_summary.json").read_text())
sensitivity = pd.read_csv(SENSITIVITY_RUN / "support_sensitivity.csv")
if set(sensitivity["policy"]) != POLICIES:
    raise RuntimeError(f"Unexpected support-policy set: {sorted(set(sensitivity['policy']))}")

for run_dir in (DECOMPOSITION_RUN, REGIME_RUN, SENSITIVITY_RUN):
    manifest = json.loads((run_dir / "run_manifest.json").read_text())
    if manifest.get("git_commit") != AORTICTURB_COMMIT:
        raise RuntimeError(
            f"Run manifest commit mismatch in {run_dir}: {manifest.get('git_commit')} != {AORTICTURB_COMMIT}"
        )
    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(f"Run did not complete cleanly: {run_dir}")

support_table = decomposition_metrics[
    [
        "phase",
        "support_fraction",
        "common_support_nodes",
        "union_support_nodes",
        "intersection_over_union_fraction",
        "masks_identical",
        "realizations",
    ]
].copy()

sensitivity_table = sensitivity[
    [
        "phase",
        "policy",
        "is_primary_policy",
        "support_evaluable",
        "support_ready",
        "projected_decision",
        "primary_decision",
        "common_intersection_fraction_total",
        "intersection_over_union_fraction",
        "fully_finite_window_count",
        "local_window_min_valid_fraction",
    ]
].copy()

display(support_table)
display(sensitivity_table)
print("Primary Gate A:", regime_summary["phases"])


,phase,support_fraction,common_support_nodes,union_support_nodes,intersection_over_union_fraction,masks_identical,realizations
0,t018,0.397379,390465,390465,1.0,True,16
1,t021,0.397379,390465,390465,1.0,True,16
2,t024,0.397379,390465,390465,1.0,True,16


,phase,policy,is_primary_policy,support_evaluable,support_ready,projected_decision,primary_decision,common_intersection_fraction_total,intersection_over_union_fraction,fully_finite_window_count,local_window_min_valid_fraction
0,t018,legacy_global_fraction,False,True,False,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
1,t018,strict_mask_consistency,False,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
2,t018,fully_finite_local_windows,False,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
3,t018,strict_mask_and_local_windows,True,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
4,t021,legacy_global_fraction,False,True,False,UNRESOLVED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0
5,t021,strict_mask_consistency,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0
6,t021,fully_finite_local_windows,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0
7,t021,strict_mask_and_local_windows,True,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0
8,t024,legacy_global_fraction,False,True,False,UNRESOLVED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0
9,t024,strict_mask_consistency,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,STOCHASTIC_MULTISCALE_SUPPORTED,0.397379,1.0,3,1.0


Primary Gate A: {'t018': {'decision': 'UNRESOLVED', 'turbulence_established': None}, 't021': {'decision': 'STOCHASTIC_MULTISCALE_SUPPORTED', 'turbulence_established': True}, 't024': {'decision': 'STOCHASTIC_MULTISCALE_SUPPORTED', 'turbulence_established': True}}


In [35]:
record_dir = WORKSPACE_ROOT / "colab_runs"
record_dir.mkdir(parents=True, exist_ok=True)
record_path = record_dir / (
    "pass3_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + ".json"
)

record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "aorticturb_commit": AORTICTURB_COMMIT,
    "dataset_root": str(DATASET_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "decomposition_run": str(DECOMPOSITION_RUN),
    "regime_run": str(REGIME_RUN),
    "support_sensitivity_run": str(SENSITIVITY_RUN),
    "primary_gate_a": regime_summary["phases"],
    "support_metrics": support_table.to_dict(orient="records"),
    "support_sensitivity": sensitivity_table.to_dict(orient="records"),
    "canonical_root_05_results_written_by_notebook": False,
}
record_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Pass-3 execution record:", record_path)
print("Canonical root 05_results written by notebook: False")


Pass-3 execution record: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/colab_runs/pass3_20261024_110057.json
Canonical root 05_results written by notebook: False


In [36]:
# ============================================================================
# NEXT LOCKED STAGE: Direct spatial Kolmogorov–Obukhov analysis
#
# Scientific scope:
#   - t018: MUST remain excluded because formal Gate A is unresolved.
#   - t021, t024: admitted because formal primary Gate A established
#     stochastic multiscale turbulence.
#   - No Taylor hypothesis.
#   - No phase pooling.
#   - No whole-domain FFT.
#   - No notebook-defined KO criteria.
#   - All KO calculations are delegated to the pinned AorticTurb package.
#
# Outputs:
#   results/04_analysis/02_kolmogorov_obukhov/run_<timestamp>/
#
# Plot-ready numerical results are retained in:
#   diagnostics.json
#   uncertainty.csv
#   metrics.csv
# plus package-generated figures under figures/.
# ============================================================================

import json
import os
import signal
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------------
# 0. LOCKED PROVENANCE CHECKS
# ---------------------------------------------------------------------------

EXPECTED_COMMIT = "43544057e627f0a00051557171fafdc2e41b7402"

if AORTICTURB_COMMIT != EXPECTED_COMMIT:
    raise RuntimeError(
        "KO_ABORT: notebook package commit is not the reviewed Pass-3/KO commit.\n"
        f"Expected: {EXPECTED_COMMIT}\n"
        f"Observed: {AORTICTURB_COMMIT}"
    )

required_paths = {
    "dataset": Path(DATASET_ROOT),
    "decomposition": Path(DECOMPOSITION_RUN),
    "regime": Path(REGIME_RUN),
    "workspace": Path(WORKSPACE_ROOT),
    "installed_package": Path(SITE_DIR),
    "python": Path(VENV_PYTHON),
}

for label, path in required_paths.items():
    if not path.exists():
        raise RuntimeError(f"KO_ABORT: missing {label}: {path}")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


# Verify upstream runs are complete and from the exact pinned commit.
for label, run_dir in (
    ("decomposition", Path(DECOMPOSITION_RUN)),
    ("Gate A", Path(REGIME_RUN)),
):
    manifest_path = run_dir / "run_manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError(f"KO_ABORT: missing {label} manifest: {manifest_path}")

    manifest = read_json(manifest_path)

    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(
            f"KO_ABORT: {label} run is not COMPLETE: {manifest.get('status')}"
        )

    if manifest.get("git_commit") != EXPECTED_COMMIT:
        raise RuntimeError(
            f"KO_ABORT: {label} commit mismatch.\n"
            f"Expected: {EXPECTED_COMMIT}\n"
            f"Observed: {manifest.get('git_commit')}"
        )


# ---------------------------------------------------------------------------
# 1. VERIFY THE FORMAL PRIMARY GATE-A HANDOFF
# ---------------------------------------------------------------------------

regime_summary_path = Path(REGIME_RUN) / "regime_summary.json"
regime_summary = read_json(regime_summary_path)

phases = regime_summary.get("phases", {})

required_gate_a = {
    "t018": None,
    "t021": True,
    "t024": True,
}

for phase, expected_turbulence in required_gate_a.items():
    if phase not in phases:
        raise RuntimeError(f"KO_ABORT: Gate A is missing {phase}")

    observed = phases[phase].get("turbulence_established")

    if observed is not expected_turbulence:
        raise RuntimeError(
            f"KO_ABORT: unexpected formal Gate-A state for {phase}: "
            f"{observed!r}; expected {expected_turbulence!r}"
        )

if phases["t018"].get("decision") != "UNRESOLVED":
    raise RuntimeError(
        "KO_ABORT: t018 must remain UNRESOLVED and must not enter KO."
    )

print("Formal Gate-A handoff verified:")
print("  t018 -> excluded (UNRESOLVED)")
print("  t021 -> admitted")
print("  t024 -> admitted")


# ---------------------------------------------------------------------------
# 2. RUN THE EXISTING AORTICTURB KO DRIVER IN A FRESH PYTHON PROCESS
#
# IMPORTANT:
#   We intentionally DO NOT supply support-sensitivity as an authorization
#   source. The package must authorize KO solely from formal primary Gate A.
# ---------------------------------------------------------------------------

ko_script = r'''
import json
import os
import sys
import traceback
from pathlib import Path

try:
    from aorticturb.config import load_config
    from aorticturb.kolmogorov.driver import run_kolmogorov

    dataset_root = Path(os.environ["AT_DATASET_ROOT"])
    decomposition_run = Path(os.environ["AT_DECOMPOSITION_RUN"])
    regime_run = Path(os.environ["AT_REGIME_RUN"])
    workspace_root = Path(os.environ["AT_WORKSPACE_ROOT"])

    config = load_config(None)

    result = run_kolmogorov(
        dataset_root,
        decomposition_run,
        regime_run,
        workspace_root,
        config,
        # Deliberately no support_sensitivity_dir.
        # Formal primary Gate A is the only KO authorization source.
    )

    print(
        "__AORTICTURB_KO_RESULT__=" + json.dumps(result),
        flush=True,
    )

except BaseException:
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise
'''

env = os.environ.copy()

# Force the fresh subprocess to import only the package installed by Cell 1.
env["PYTHONPATH"] = str(Path(SITE_DIR)) + (
    os.pathsep + env["PYTHONPATH"]
    if env.get("PYTHONPATH")
    else ""
)

env["AT_DATASET_ROOT"] = str(Path(DATASET_ROOT).resolve())
env["AT_DECOMPOSITION_RUN"] = str(Path(DECOMPOSITION_RUN).resolve())
env["AT_REGIME_RUN"] = str(Path(REGIME_RUN).resolve())
env["AT_WORKSPACE_ROOT"] = str(Path(WORKSPACE_ROOT).resolve())

env["PYTHONUNBUFFERED"] = "1"

# Avoid uncontrolled numerical-library thread pools in Colab.
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"

print("\nStarting direct spatial KO analysis...\n")

process = subprocess.Popen(
    [str(VENV_PYTHON), "-c", ko_script],
    cwd="/content",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

return_code = process.wait()

if return_code != 0:
    if return_code == -signal.SIGKILL or return_code == 137:
        raise RuntimeError(
            "KO_PROCESS_KILLED: the KO process was killed by SIGKILL. "
            "This normally indicates a Colab memory/resource kill."
        )

    if return_code == -signal.SIGSEGV or return_code == 139:
        raise RuntimeError(
            "KO_PROCESS_CRASHED: the KO process terminated with SIGSEGV."
        )

    raise RuntimeError(
        f"KO_FAILED: process exited with code {return_code}. "
        "See the streamed output above."
    )


# ---------------------------------------------------------------------------
# 3. RECOVER THE EXACT KO RUN DIRECTORY
# ---------------------------------------------------------------------------

prefix = "__AORTICTURB_KO_RESULT__="

result_lines = [
    line.strip()
    for line in output_lines
    if line.startswith(prefix)
]

if len(result_lines) != 1:
    raise RuntimeError(
        "KO_ABORT: process completed but did not return exactly one result record."
    )

ko_result = json.loads(result_lines[0][len(prefix):])
KO_RUN = Path(ko_result["run_dir"]).resolve()

if not KO_RUN.is_dir():
    raise RuntimeError(f"KO_ABORT: KO run directory does not exist: {KO_RUN}")


# ---------------------------------------------------------------------------
# 4. VERIFY COMPLETE, PROVENANCE-VALID OUTPUT
# ---------------------------------------------------------------------------

required_outputs = (
    "run_manifest.json",
    "kolmogorov_summary.json",
    "diagnostics.json",
    "metrics.csv",
    "uncertainty.csv",
    "methods.md",
)

missing = [
    name
    for name in required_outputs
    if not (KO_RUN / name).is_file()
]

if missing:
    raise RuntimeError(
        "KO_ABORT: required KO outputs are missing: "
        + ", ".join(missing)
    )

ko_manifest = read_json(KO_RUN / "run_manifest.json")

if ko_manifest.get("status") != "COMPLETE":
    raise RuntimeError(
        f"KO_ABORT: KO manifest status is {ko_manifest.get('status')!r}"
    )

if ko_manifest.get("git_commit") != EXPECTED_COMMIT:
    raise RuntimeError(
        "KO_ABORT: KO run was not produced by the pinned scientific commit."
    )

ko_summary = read_json(KO_RUN / "kolmogorov_summary.json")
ko_phases = ko_summary.get("phases", {})


# ---------------------------------------------------------------------------
# 5. ENFORCE THE LOCKED PHASE HANDOFF — NOT THE KO OUTCOME
#
# We check only which phases were analyzed.
# We DO NOT require KO_SUPPORTED or KO_INCONSISTENT.
# The scientific outcome must come from the diagnostics.
# ---------------------------------------------------------------------------

if ko_phases.get("t018", {}).get("status") != "SKIPPED_GATE_A":
    raise RuntimeError(
        "KO_ABORT: t018 entered KO despite unresolved formal Gate A."
    )

for phase in ("t021", "t024"):
    status = ko_phases.get(phase, {}).get("status")

    if status in (None, "SKIPPED_GATE_A"):
        raise RuntimeError(
            f"KO_ABORT: {phase} was not actually analyzed by the KO stage."
        )

    handoff = ko_phases[phase].get("gate_a_handoff", {})

    if handoff.get("source") != "primary_gate_a":
        raise RuntimeError(
            f"KO_ABORT: {phase} was not authorized by primary Gate A."
        )

    if handoff.get("ko_authorized") is not True:
        raise RuntimeError(
            f"KO_ABORT: primary Gate-A authorization missing for {phase}."
        )


# ---------------------------------------------------------------------------
# 6. DISPLAY RESULTS AND RECORD THE RUN POINTER
# ---------------------------------------------------------------------------

ko_metrics = pd.read_csv(KO_RUN / "metrics.csv")
ko_uncertainty = pd.read_csv(KO_RUN / "uncertainty.csv")

print("\nKO analysis completed successfully.")
print("KO run:", KO_RUN)
print("Scientific commit:", ko_manifest.get("git_commit"))

print("\nPhase summary:")
for phase in ("t018", "t021", "t024"):
    print(f"  {phase}: {ko_phases.get(phase)}")

print("\nKO metrics:")
display(ko_metrics)

print("\nUncertainty table:")
display(ko_uncertainty)

print("\nPlot-ready sources:")
print(" ", KO_RUN / "diagnostics.json")
print(" ", KO_RUN / "uncertainty.csv")
print(" ", KO_RUN / "metrics.csv")
print(" ", KO_RUN / "figures")


# Preserve a notebook-level pointer without modifying the finalized KO run.
record_dir = Path(WORKSPACE_ROOT) / "colab_runs"
record_dir.mkdir(parents=True, exist_ok=True)

record_path = record_dir / (
    "ko_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + ".json"
)

record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "aorticturb_commit": EXPECTED_COMMIT,
    "dataset_root": str(Path(DATASET_ROOT).resolve()),
    "decomposition_run": str(Path(DECOMPOSITION_RUN).resolve()),
    "regime_run": str(Path(REGIME_RUN).resolve()),
    "kolmogorov_run": str(KO_RUN),
    "formal_gate_a": phases,
    "kolmogorov_summary": ko_phases,
    "canonical_root_05_results_written_by_notebook": False,
}

record_path.write_text(
    json.dumps(record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\nKO execution record:", record_path)
print("Canonical root 05_results written by this cell: False")

Formal Gate-A handoff verified:
  t018 -> excluded (UNRESOLVED)
  t021 -> admitted
  t024 -> admitted

Starting direct spatial KO analysis...

__AORTICTURB_KO_RESULT__={"run_dir": "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/02_kolmogorov_obukhov/run_20261024_110829_563323", "phases": {"t018": {"gate_a_handoff": {"ko_authorized": false, "phase": "t018", "primary_decision": "UNRESOLVED", "primary_turbulence_established": null, "reason": "Primary Gate A is unresolved.", "source": "primary_gate_a"}, "ko_resolved": null, "ko_supported": null, "status": "SKIPPED_GATE_A"}, "t021": {"core_diagnostics": {"S2": "INCONSISTENT", "S3_four_fifths": "UNRESOLVED", "flux_plateau": "UNRESOLVED", "spectrum": "INCONSISTENT"}, "gate_a_handoff": {"ko_authorized": true, "phase": "t021", "primary_decision": "STOCHASTIC_MULTISCALE_SUPPORTED", "primary_turbulence_established": true, "reason": "Primary Gate A established stochastic multiscale turbulence.", "source": "primar

,phase,status,ko_resolved,ko_supported,gate_a_handoff_authorized,gate_a_handoff_source,local_isotropy_admissible,status_spectrum,status_S2,status_S3_four_fifths,status_flux_plateau,local_homogeneity_not_rejected
0,t018,SKIPPED_GATE_A,NaN,NaN,False,primary_gate_a,NaN,NaN,NaN,NaN,NaN,NaN
1,t021,KO_UNRESOLVED,False,NaN,True,primary_gate_a,False,INCONSISTENT,INCONSISTENT,UNRESOLVED,UNRESOLVED,False
2,t024,KO_UNRESOLVED,False,NaN,True,primary_gate_a,False,INCONSISTENT,INCONSISTENT,UNRESOLVED,INCONSISTENT,False



Uncertainty table:


,phase,direction,quantity,lower,median,upper,theory_target,bootstrap_axis
0,t021,x,spectrum_slope,-4.377865,-4.245715,-4.129056,-1.666667,realizations
1,t021,x,S2_slope,1.230416,1.251050,1.270028,0.666667,realizations
2,t021,x,S3_slope,1.759410,2.212591,2.635672,1.000000,realizations
3,t021,x,four_fifths_ratio,-2.701287,0.362459,3.010838,1.000000,realizations
4,t021,x,ESS_zeta4,0.812335,0.965280,1.131035,NaN,realizations
5,t021,x,ESS_zeta6,1.071862,1.300767,1.533402,NaN,realizations
6,t021,y,spectrum_slope,-4.415644,-4.306394,-4.174071,-1.666667,realizations
7,t021,y,S2_slope,1.262077,1.292585,1.324549,0.666667,realizations
8,t021,y,S3_slope,1.699331,2.223680,2.807104,1.000000,realizations
9,t021,y,four_fifths_ratio,-11.866963,-2.888009,4.790578,1.000000,realizations



Plot-ready sources:
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/02_kolmogorov_obukhov/run_20261024_110829_563323/diagnostics.json
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/02_kolmogorov_obukhov/run_20261024_110829_563323/uncertainty.csv
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/02_kolmogorov_obukhov/run_20261024_110829_563323/metrics.csv
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/02_kolmogorov_obukhov/run_20261024_110829_563323/figures

KO execution record: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/colab_runs/ko_20261024_110910.json
Canonical root 05_results written by this cell: False


In [37]:
# ============================================================================
# NEXT LOCKED STAGE: Taylor-hypothesis admissibility
#
# Scientific scope:
#   - Runs only after the completed KO stage.
#   - t018 remains excluded because formal Gate A is unresolved.
#   - t021 and t024 are assessed because turbulence is established.
#   - NO concatenation of phases.
#   - NO concatenation of realizations into a time series.
#   - NO frequency spectrum.
#   - NO Taylor f -> k conversion.
#   - NO notebook-defined Taylor threshold or classification.
#
# The pinned AorticTurb package assesses only the necessary conditions
# available from this dataset:
#   chi = u_rms' / |U_bar|
#   fraction of valid nodes below the package's existing chi warning value
#   phase-mean directional alignment
#   phase-mean speed coefficient of variation
#
# Outputs:
#   results/04_analysis/03_taylor_hypothesis/run_<timestamp>/
#
# Plot-ready numerical outputs:
#   diagnostics.json
#   metrics.csv
#   uncertainty.csv
# ============================================================================

import json
import os
import signal
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------------
# 0. LOCKED PROVENANCE
# ---------------------------------------------------------------------------

EXPECTED_COMMIT = "43544057e627f0a00051557171fafdc2e41b7402"

if AORTICTURB_COMMIT != EXPECTED_COMMIT:
    raise RuntimeError(
        "TAYLOR_ABORT: notebook package commit is not the reviewed scientific commit.\n"
        f"Expected: {EXPECTED_COMMIT}\n"
        f"Observed: {AORTICTURB_COMMIT}"
    )

required_paths = {
    "dataset": Path(DATASET_ROOT),
    "decomposition": Path(DECOMPOSITION_RUN),
    "Gate A": Path(REGIME_RUN),
    "KO": Path(KO_RUN),
    "workspace": Path(WORKSPACE_ROOT),
    "installed package": Path(SITE_DIR),
    "python": Path(VENV_PYTHON),
}

for label, path in required_paths.items():
    if not path.exists():
        raise RuntimeError(f"TAYLOR_ABORT: missing {label}: {path}")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


# ---------------------------------------------------------------------------
# 1. VERIFY UPSTREAM SCIENTIFIC SEQUENCE
# ---------------------------------------------------------------------------

for label, run_dir in (
    ("decomposition", Path(DECOMPOSITION_RUN)),
    ("Gate A", Path(REGIME_RUN)),
    ("KO", Path(KO_RUN)),
):
    manifest_path = run_dir / "run_manifest.json"

    if not manifest_path.is_file():
        raise RuntimeError(
            f"TAYLOR_ABORT: missing {label} manifest: {manifest_path}"
        )

    manifest = read_json(manifest_path)

    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(
            f"TAYLOR_ABORT: {label} run is not COMPLETE: "
            f"{manifest.get('status')!r}"
        )

    if manifest.get("git_commit") != EXPECTED_COMMIT:
        raise RuntimeError(
            f"TAYLOR_ABORT: {label} commit mismatch.\n"
            f"Expected: {EXPECTED_COMMIT}\n"
            f"Observed: {manifest.get('git_commit')}"
        )


# Formal Gate-A state must remain exactly the established state.
regime_summary = read_json(
    Path(REGIME_RUN) / "regime_summary.json"
)

gate_phases = regime_summary.get("phases", {})

expected_gate = {
    "t018": None,
    "t021": True,
    "t024": True,
}

for phase, expected in expected_gate.items():
    observed = gate_phases.get(phase, {}).get(
        "turbulence_established"
    )

    if observed is not expected:
        raise RuntimeError(
            f"TAYLOR_ABORT: unexpected Gate-A state for {phase}: "
            f"{observed!r}; expected {expected!r}"
        )


# Verify that KO has already been completed in the locked sequence.
ko_summary = read_json(
    Path(KO_RUN) / "kolmogorov_summary.json"
)
ko_phases = ko_summary.get("phases", {})

if ko_phases.get("t018", {}).get("status") != "SKIPPED_GATE_A":
    raise RuntimeError(
        "TAYLOR_ABORT: t018 did not remain excluded from KO."
    )

for phase in ("t021", "t024"):
    ko_status = ko_phases.get(phase, {}).get("status")

    if ko_status in (None, "SKIPPED_GATE_A"):
        raise RuntimeError(
            f"TAYLOR_ABORT: {phase} did not complete the KO stage."
        )

print("Locked upstream sequence verified:")
print("  t018 -> Gate A unresolved; excluded")
print(f"  t021 -> turbulence established; KO status = "
      f"{ko_phases['t021']['status']}")
print(f"  t024 -> turbulence established; KO status = "
      f"{ko_phases['t024']['status']}")


# ---------------------------------------------------------------------------
# 2. RUN THE PINNED AORTICTURB TAYLOR DRIVER
#
# No temporal series is created here.
# No phase or realization concatenation occurs.
# ---------------------------------------------------------------------------

taylor_script = r'''
import json
import os
import traceback
from pathlib import Path

try:
    from aorticturb.config import load_config
    from aorticturb.taylor.driver import run_taylor

    dataset_root = Path(os.environ["AT_DATASET_ROOT"])
    decomposition_run = Path(os.environ["AT_DECOMPOSITION_RUN"])
    regime_run = Path(os.environ["AT_REGIME_RUN"])
    workspace_root = Path(os.environ["AT_WORKSPACE_ROOT"])

    config = load_config(None)

    result = run_taylor(
        dataset_root,
        decomposition_run,
        regime_run,
        workspace_root,
        config,
    )

    print(
        "__AORTICTURB_TAYLOR_RESULT__=" + json.dumps(result),
        flush=True,
    )

except BaseException:
    traceback.print_exc()
    raise
'''

env = os.environ.copy()

env["PYTHONPATH"] = str(Path(SITE_DIR)) + (
    os.pathsep + env["PYTHONPATH"]
    if env.get("PYTHONPATH")
    else ""
)

env["AT_DATASET_ROOT"] = str(
    Path(DATASET_ROOT).resolve()
)
env["AT_DECOMPOSITION_RUN"] = str(
    Path(DECOMPOSITION_RUN).resolve()
)
env["AT_REGIME_RUN"] = str(
    Path(REGIME_RUN).resolve()
)
env["AT_WORKSPACE_ROOT"] = str(
    Path(WORKSPACE_ROOT).resolve()
)

env["PYTHONUNBUFFERED"] = "1"

# Keep numerical-library threading controlled in Colab.
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"

print("\nStarting Taylor admissibility analysis...\n")

process = subprocess.Popen(
    [str(VENV_PYTHON), "-c", taylor_script],
    cwd="/content",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

return_code = process.wait()

if return_code != 0:
    if return_code in (-signal.SIGKILL, 137):
        raise RuntimeError(
            "TAYLOR_PROCESS_KILLED: subprocess was killed."
        )

    if return_code in (-signal.SIGSEGV, 139):
        raise RuntimeError(
            "TAYLOR_PROCESS_CRASHED: subprocess terminated "
            "with SIGSEGV."
        )

    raise RuntimeError(
        f"TAYLOR_FAILED: subprocess exited with "
        f"code {return_code}."
    )


# ---------------------------------------------------------------------------
# 3. RECOVER EXACT OUTPUT DIRECTORY
# ---------------------------------------------------------------------------

prefix = "__AORTICTURB_TAYLOR_RESULT__="

result_lines = [
    line.strip()
    for line in output_lines
    if line.startswith(prefix)
]

if len(result_lines) != 1:
    raise RuntimeError(
        "TAYLOR_ABORT: subprocess completed but did not "
        "return exactly one result record."
    )

taylor_result = json.loads(
    result_lines[0][len(prefix):]
)

TAYLOR_RUN = Path(
    taylor_result["run_dir"]
).resolve()

if not TAYLOR_RUN.is_dir():
    raise RuntimeError(
        f"TAYLOR_ABORT: run directory missing: {TAYLOR_RUN}"
    )


# ---------------------------------------------------------------------------
# 4. VERIFY REQUIRED OUTPUTS
# ---------------------------------------------------------------------------

required_outputs = (
    "run_manifest.json",
    "taylor_summary.json",
    "diagnostics.json",
    "metrics.csv",
    "uncertainty.csv",
    "methods.md",
)

missing = [
    name
    for name in required_outputs
    if not (TAYLOR_RUN / name).is_file()
]

if missing:
    raise RuntimeError(
        "TAYLOR_ABORT: missing required outputs: "
        + ", ".join(missing)
    )

taylor_manifest = read_json(
    TAYLOR_RUN / "run_manifest.json"
)

if taylor_manifest.get("status") != "COMPLETE":
    raise RuntimeError(
        "TAYLOR_ABORT: Taylor run did not finalize COMPLETE."
    )

if taylor_manifest.get("git_commit") != EXPECTED_COMMIT:
    raise RuntimeError(
        "TAYLOR_ABORT: Taylor run commit mismatch."
    )


# ---------------------------------------------------------------------------
# 5. VERIFY ONLY THE LOCKED DATA AVAILABILITY LOGIC
#
# Do NOT enforce a desired Taylor outcome.
# ---------------------------------------------------------------------------

taylor_summary = read_json(
    TAYLOR_RUN / "taylor_summary.json"
)

taylor_phases = taylor_summary.get("phases", {})

if taylor_summary.get("temporal_test_available") is not False:
    raise RuntimeError(
        "TAYLOR_ABORT: this dataset must not be treated as "
        "providing a continuous temporal Taylor test."
    )

if (
    taylor_phases.get("t018", {}).get("status")
    != "SKIPPED_GATE_A"
):
    raise RuntimeError(
        "TAYLOR_ABORT: t018 entered Taylor analysis despite "
        "unresolved Gate A."
    )

expected_assessed_status = (
    "NECESSARY_CONDITIONS_ASSESSED_TEMPORAL_TEST_UNAVAILABLE"
)

for phase in ("t021", "t024"):
    status = taylor_phases.get(phase, {}).get("status")

    if status != expected_assessed_status:
        raise RuntimeError(
            f"TAYLOR_ABORT: unexpected Taylor-stage state "
            f"for {phase}: {status!r}"
        )


# ---------------------------------------------------------------------------
# 6. DISPLAY AND PRESERVE PLOT-READY RESULTS
# ---------------------------------------------------------------------------

taylor_metrics = pd.read_csv(
    TAYLOR_RUN / "metrics.csv"
)

taylor_uncertainty = pd.read_csv(
    TAYLOR_RUN / "uncertainty.csv"
)

taylor_diagnostics = read_json(
    TAYLOR_RUN / "diagnostics.json"
)

print("\nTaylor admissibility stage completed.")
print("Taylor run:", TAYLOR_RUN)
print(
    "Scientific commit:",
    taylor_manifest.get("git_commit"),
)

print("\nTaylor metrics:")
display(taylor_metrics)

print("\nTaylor uncertainty:")
display(taylor_uncertainty)

print("\nImportant interpretation constraint:")
print("  temporal_test_available = False")
print(
    "  This stage assesses necessary spatial/mean-flow "
    "conditions only."
)
print(
    "  No Taylor frequency-to-wavenumber conversion "
    "is permitted."
)

print("\nPlot-ready sources:")
print(" ", TAYLOR_RUN / "diagnostics.json")
print(" ", TAYLOR_RUN / "metrics.csv")
print(" ", TAYLOR_RUN / "uncertainty.csv")


# ---------------------------------------------------------------------------
# 7. NOTEBOOK-LEVEL EXECUTION POINTER
#
# Does not modify the finalized Taylor run.
# Does not promote anything to canonical 05_results.
# ---------------------------------------------------------------------------

record_dir = (
    Path(WORKSPACE_ROOT) / "colab_runs"
)
record_dir.mkdir(
    parents=True,
    exist_ok=True,
)

record_path = record_dir / (
    "taylor_"
    + datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".json"
)

record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "aorticturb_commit": EXPECTED_COMMIT,
    "dataset_root": str(
        Path(DATASET_ROOT).resolve()
    ),
    "decomposition_run": str(
        Path(DECOMPOSITION_RUN).resolve()
    ),
    "regime_run": str(
        Path(REGIME_RUN).resolve()
    ),
    "kolmogorov_run": str(
        Path(KO_RUN).resolve()
    ),
    "taylor_run": str(TAYLOR_RUN),
    "formal_gate_a": {
        phase: {
            "decision": gate_phases[phase].get("decision"),
            "turbulence_established":
                gate_phases[phase].get(
                    "turbulence_established"
                ),
        }
        for phase in ("t018", "t021", "t024")
    },
    "ko_status": {
        phase: ko_phases[phase].get("status")
        for phase in ("t018", "t021", "t024")
    },
    "taylor_summary": taylor_phases,
    "temporal_test_available": False,
    "canonical_root_05_results_written_by_notebook": False,
}

record_path.write_text(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "\nTaylor execution record:",
    record_path,
)
print(
    "Canonical root 05_results written by this cell: False"
)

Locked upstream sequence verified:
  t018 -> Gate A unresolved; excluded
  t021 -> turbulence established; KO status = KO_UNRESOLVED
  t024 -> turbulence established; KO status = KO_UNRESOLVED

Starting Taylor admissibility analysis...

/content/aorticturb-pass3-site/aorticturb/taylor/admissibility.py:20: RuntimeWarning: Mean of empty slice
  u_rms = np.sqrt(np.nanmean(np.sum(residuals**2, axis=-1), axis=0))
__AORTICTURB_TAYLOR_RESULT__={"run_dir": "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/03_taylor_hypothesis/run_20261024_111454_006245", "phases": {"t018": {"status": "SKIPPED_GATE_A", "temporal_test_available": false}, "t021": {"status": "NECESSARY_CONDITIONS_ASSESSED_TEMPORAL_TEST_UNAVAILABLE", "temporal_test_available": false, "necessary_spatial_assessment": {"median_fluctuation_ratio": 0.7401578418379767, "fraction_below_threshold": 0.11972080306589451, "temporal_test_available": false, "conclusion": "NECESSARY_SPATIAL_ADMISSIBILITY_ONLY: th

,phase,status,temporal_test_available,median_chi,fraction_chi_below_warning,chi_warning_value,mean_direction_alignment,mean_speed_cv
0,t018,SKIPPED_GATE_A,False,NaN,NaN,NaN,NaN,NaN
1,t021,NECESSARY_CONDITIONS_ASSESSED_TEMPORAL_TEST_UN...,False,0.740158,0.119721,0.2,0.599211,0.797095
2,t024,NECESSARY_CONDITIONS_ASSESSED_TEMPORAL_TEST_UN...,False,0.799108,0.099105,0.2,0.567168,0.807607



Taylor uncertainty:


,phase,quantity,lower,median,upper,bootstrap_axis
0,t021,realization_median_chi,0.600428,0.609039,0.619361,realizations
1,t024,realization_median_chi,0.648644,0.657170,0.665172,realizations



Important interpretation constraint:
  temporal_test_available = False
  This stage assesses necessary spatial/mean-flow conditions only.
  No Taylor frequency-to-wavenumber conversion is permitted.

Plot-ready sources:
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/03_taylor_hypothesis/run_20261024_111454_006245/diagnostics.json
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/03_taylor_hypothesis/run_20261024_111454_006245/metrics.csv
  /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/03_taylor_hypothesis/run_20261024_111454_006245/uncertainty.csv

Taylor execution record: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/colab_runs/taylor_20261024_111504.json
Canonical root 05_results written by this cell: False


In [39]:
# ============================================================================
# NEXT LOCKED STAGE: Lumley / Reynolds-stress anisotropy analysis
#
# Scientific scope:
#   - t018 remains excluded because formal Gate A is unresolved.
#   - t021 and t024 are characterized because turbulence is established.
#   - No isotropy assumption is introduced.
#   - No KO or Taylor result is modified or reinterpreted.
#   - No new threshold or classification rule is introduced.
#
# Existing definitions:
#
#   b_ij = R_ij/(2k) - delta_ij/3
#
#   II  = -0.5 b_ij b_ji
#   III = (1/3) b_ij b_jk b_ki
#
# In addition to phase summaries, the upgraded AorticTurb driver persists
# full valid-node plot-ready anisotropy fields:
#
#   node_index
#   coordinates
#   k
#   b
#   II
#   III
#   eigenvalues
#   barycentric
#   realizable
#
# Output:
#   results/04_analysis/04_lumley_anisotropy/run_<timestamp>/
#
# This stage uses the persistence-only AorticTurb upgrade merged in PR #10.
# Previous decomposition / Gate-A / KO / Taylor runs remain immutable.
# ============================================================================

import json
import os
import shutil
import signal
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ---------------------------------------------------------------------------
# 0. LOCKED COMMITS
# ---------------------------------------------------------------------------

# Commit used by the already completed scientific analyses.
UPSTREAM_COMMIT = "43544057e627f0a00051557171fafdc2e41b7402"

# Merged main commit containing ONLY the Lumley plot-data persistence upgrade.
LUMLEY_COMMIT = "c74cdd8ca072f426a118b30df9ea98d8ae7372a2"

AORTICTURB_REPO = "https://github.com/khalid-saqr/AorticTurb.git"


# ---------------------------------------------------------------------------
# 1. VERIFY NOTEBOOK STATE
# ---------------------------------------------------------------------------

required_variables = (
    "DATASET_ROOT",
    "DECOMPOSITION_RUN",
    "REGIME_RUN",
    "KO_RUN",
    "TAYLOR_RUN",
    "WORKSPACE_ROOT",
    "VENV_PYTHON",
)

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "LUMLEY_ABORT: required notebook variables are missing: "
        + ", ".join(missing_variables)
    )


required_paths = {
    "dataset": Path(DATASET_ROOT),
    "decomposition": Path(DECOMPOSITION_RUN),
    "Gate A": Path(REGIME_RUN),
    "KO": Path(KO_RUN),
    "Taylor": Path(TAYLOR_RUN),
    "workspace": Path(WORKSPACE_ROOT),
    "python": Path(VENV_PYTHON),
}

for label, path in required_paths.items():
    if not path.exists():
        raise RuntimeError(
            f"LUMLEY_ABORT: missing {label}: {path}"
        )


def read_json(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


# ---------------------------------------------------------------------------
# 2. VERIFY IMMUTABLE UPSTREAM PROVENANCE
# ---------------------------------------------------------------------------

for label, run_dir in (
    ("decomposition", Path(DECOMPOSITION_RUN)),
    ("Gate A", Path(REGIME_RUN)),
    ("KO", Path(KO_RUN)),
    ("Taylor", Path(TAYLOR_RUN)),
):
    manifest_path = run_dir / "run_manifest.json"

    if not manifest_path.is_file():
        raise RuntimeError(
            f"LUMLEY_ABORT: missing {label} manifest: "
            f"{manifest_path}"
        )

    manifest = read_json(manifest_path)

    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(
            f"LUMLEY_ABORT: {label} run is not COMPLETE: "
            f"{manifest.get('status')!r}"
        )

    if manifest.get("git_commit") != UPSTREAM_COMMIT:
        raise RuntimeError(
            f"LUMLEY_ABORT: unexpected {label} scientific commit.\n"
            f"Expected: {UPSTREAM_COMMIT}\n"
            f"Observed: {manifest.get('git_commit')}"
        )


# ---------------------------------------------------------------------------
# 3. VERIFY LOCKED SCIENTIFIC HANDOFF
# ---------------------------------------------------------------------------

regime_summary = read_json(
    Path(REGIME_RUN) / "regime_summary.json"
)
gate_phases = regime_summary.get("phases", {})

expected_gate = {
    "t018": None,
    "t021": True,
    "t024": True,
}

for phase, expected in expected_gate.items():
    observed = gate_phases.get(
        phase, {}
    ).get("turbulence_established")

    if observed is not expected:
        raise RuntimeError(
            f"LUMLEY_ABORT: unexpected Gate-A state "
            f"for {phase}: {observed!r}; expected {expected!r}"
        )

if gate_phases["t018"].get("decision") != "UNRESOLVED":
    raise RuntimeError(
        "LUMLEY_ABORT: t018 must remain unresolved."
    )


# Verify completed KO stage.
ko_summary = read_json(
    Path(KO_RUN) / "kolmogorov_summary.json"
)
ko_phases = ko_summary.get("phases", {})

if ko_phases.get("t018", {}).get("status") != "SKIPPED_GATE_A":
    raise RuntimeError(
        "LUMLEY_ABORT: t018 did not remain excluded from KO."
    )

for phase in ("t021", "t024"):
    if ko_phases.get(phase, {}).get("status") != "KO_UNRESOLVED":
        raise RuntimeError(
            f"LUMLEY_ABORT: unexpected recorded KO state "
            f"for {phase}: "
            f"{ko_phases.get(phase, {}).get('status')!r}"
        )


# Verify completed Taylor stage.
taylor_summary = read_json(
    Path(TAYLOR_RUN) / "taylor_summary.json"
)
taylor_phases = taylor_summary.get("phases", {})

if taylor_summary.get("temporal_test_available") is not False:
    raise RuntimeError(
        "LUMLEY_ABORT: Taylor temporal-test state is unexpected."
    )

if (
    taylor_phases.get("t018", {}).get("status")
    != "SKIPPED_GATE_A"
):
    raise RuntimeError(
        "LUMLEY_ABORT: t018 did not remain excluded from Taylor."
    )

expected_taylor_status = (
    "NECESSARY_CONDITIONS_ASSESSED_TEMPORAL_TEST_UNAVAILABLE"
)

for phase in ("t021", "t024"):
    if (
        taylor_phases.get(phase, {}).get("status")
        != expected_taylor_status
    ):
        raise RuntimeError(
            f"LUMLEY_ABORT: unexpected Taylor-stage state "
            f"for {phase}."
        )


print("Locked upstream sequence verified:")
print("  t018 -> Gate A unresolved; excluded")
print("  t021 -> turbulence established; KO unresolved; Taylor not established")
print("  t024 -> turbulence established; KO unresolved; Taylor not established")


# ---------------------------------------------------------------------------
# 4. INSTALL THE PERSISTENCE-UPGRADED AORTICTURB COMMIT
#
# IMPORTANT:
#   This does NOT replace the package used for the completed upstream runs.
#   It creates a separate scratch installation used only by this stage.
# ---------------------------------------------------------------------------

LUMLEY_SITE_DIR = Path(
    f"/content/aorticturb-lumley-site-{LUMLEY_COMMIT[:12]}"
)

# Clean only this disposable stage-specific installation.
if LUMLEY_SITE_DIR.exists():
    shutil.rmtree(LUMLEY_SITE_DIR)

LUMLEY_SITE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nInstalling exact Lumley persistence commit:")
print(" ", LUMLEY_COMMIT)

install_cmd = [
    str(VENV_PYTHON),
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--quiet",
    "--no-deps",
    "--target",
    str(LUMLEY_SITE_DIR),
    f"git+{AORTICTURB_REPO}@{LUMLEY_COMMIT}",
]

install_result = subprocess.run(
    install_cmd,
    cwd="/content",
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if install_result.returncode != 0:
    print(install_result.stdout)
    raise RuntimeError(
        "LUMLEY_ABORT: failed to install the exact "
        "persistence-upgraded AorticTurb commit."
    )


# ---------------------------------------------------------------------------
# 5. VERIFY THE ACTUAL INSTALLED COMMIT BEFORE ANALYSIS
# ---------------------------------------------------------------------------

lumley_env = os.environ.copy()

lumley_env["PYTHONPATH"] = str(LUMLEY_SITE_DIR) + (
    os.pathsep + lumley_env["PYTHONPATH"]
    if lumley_env.get("PYTHONPATH")
    else ""
)

lumley_env["PYTHONUNBUFFERED"] = "1"
lumley_env["OMP_NUM_THREADS"] = "1"
lumley_env["OPENBLAS_NUM_THREADS"] = "1"
lumley_env["MKL_NUM_THREADS"] = "1"
lumley_env["NUMEXPR_NUM_THREADS"] = "1"


probe_script = r'''
import json
from importlib import metadata
import aorticturb

dist = metadata.distribution("aorticturb")
raw = dist.read_text("direct_url.json")

if raw is None:
    raise RuntimeError("Missing PEP-610 direct_url.json")

payload = json.loads(raw)
vcs = payload.get("vcs_info", {})
commit = vcs.get("commit_id")

print(
    "__AORTICTURB_LUMLEY_PROBE__="
    + json.dumps(
        {
            "commit": commit,
            "module_file": aorticturb.__file__,
            "version": metadata.version("aorticturb"),
        }
    ),
    flush=True,
)
'''

probe = subprocess.run(
    [str(VENV_PYTHON), "-c", probe_script],
    cwd="/content",
    env=lumley_env,
    capture_output=True,
    text=True,
)

if probe.returncode != 0:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError(
        "LUMLEY_ABORT: unable to verify installed package provenance."
    )

probe_prefix = "__AORTICTURB_LUMLEY_PROBE__="

probe_lines = [
    line
    for line in probe.stdout.splitlines()
    if line.startswith(probe_prefix)
]

if len(probe_lines) != 1:
    raise RuntimeError(
        "LUMLEY_ABORT: ambiguous package provenance probe."
    )

probe_info = json.loads(
    probe_lines[0][len(probe_prefix):]
)

if probe_info.get("commit") != LUMLEY_COMMIT:
    raise RuntimeError(
        "LUMLEY_ABORT: installed package commit mismatch.\n"
        f"Expected: {LUMLEY_COMMIT}\n"
        f"Observed: {probe_info.get('commit')}"
    )

module_path = Path(
    probe_info["module_file"]
).resolve()

if LUMLEY_SITE_DIR.resolve() not in module_path.parents:
    raise RuntimeError(
        "LUMLEY_ABORT: Python imported AorticTurb from the "
        "wrong installation."
    )

print("Lumley-stage package provenance verified:")
print("  commit :", probe_info["commit"])
print("  version:", probe_info["version"])
print("  module :", probe_info["module_file"])


# ---------------------------------------------------------------------------
# 6. RUN THE EXISTING AORTICTURB LUMLEY DRIVER
#
# The notebook defines NO anisotropy mathematics.
# Everything is delegated to the reviewed package implementation.
# ---------------------------------------------------------------------------

lumley_script = r'''
import json
import os
import traceback
from pathlib import Path

try:
    from aorticturb.config import load_config
    from aorticturb.anisotropy.driver import (
        PLOT_DATA_FIELDS,
        run_anisotropy,
    )

    dataset_root = Path(os.environ["AT_DATASET_ROOT"])
    decomposition_run = Path(os.environ["AT_DECOMPOSITION_RUN"])
    regime_run = Path(os.environ["AT_REGIME_RUN"])
    workspace_root = Path(os.environ["AT_WORKSPACE_ROOT"])

    config = load_config(None)

    result = run_anisotropy(
        dataset_root,
        decomposition_run,
        regime_run,
        workspace_root,
        config,
    )

    print(
        "__AORTICTURB_LUMLEY_RESULT__="
        + json.dumps(
            {
                "result": result,
                "plot_data_fields": list(PLOT_DATA_FIELDS),
            }
        ),
        flush=True,
    )

except BaseException:
    traceback.print_exc()
    raise
'''


lumley_env["AT_DATASET_ROOT"] = str(
    Path(DATASET_ROOT).resolve()
)
lumley_env["AT_DECOMPOSITION_RUN"] = str(
    Path(DECOMPOSITION_RUN).resolve()
)
lumley_env["AT_REGIME_RUN"] = str(
    Path(REGIME_RUN).resolve()
)
lumley_env["AT_WORKSPACE_ROOT"] = str(
    Path(WORKSPACE_ROOT).resolve()
)


print("\nStarting Lumley/Reynolds-stress anisotropy analysis...\n")

process = subprocess.Popen(
    [str(VENV_PYTHON), "-c", lumley_script],
    cwd="/content",
    env=lumley_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

return_code = process.wait()

if return_code != 0:
    if return_code in (-signal.SIGKILL, 137):
        raise RuntimeError(
            "LUMLEY_PROCESS_KILLED: subprocess was killed, "
            "most likely by a runtime resource limit."
        )

    if return_code in (-signal.SIGSEGV, 139):
        raise RuntimeError(
            "LUMLEY_PROCESS_CRASHED: subprocess terminated "
            "with SIGSEGV."
        )

    raise RuntimeError(
        f"LUMLEY_FAILED: subprocess exited with code "
        f"{return_code}."
    )


# ---------------------------------------------------------------------------
# 7. RECOVER EXACT RUN DIRECTORY
# ---------------------------------------------------------------------------

result_prefix = "__AORTICTURB_LUMLEY_RESULT__="

result_lines = [
    line.strip()
    for line in output_lines
    if line.startswith(result_prefix)
]

if len(result_lines) != 1:
    raise RuntimeError(
        "LUMLEY_ABORT: subprocess completed but did not return "
        "exactly one result record."
    )

result_payload = json.loads(
    result_lines[0][len(result_prefix):]
)

lumley_result = result_payload["result"]
EXPECTED_PLOT_FIELDS = tuple(
    result_payload["plot_data_fields"]
)

LUMLEY_RUN = Path(
    lumley_result["run_dir"]
).resolve()

if not LUMLEY_RUN.is_dir():
    raise RuntimeError(
        f"LUMLEY_ABORT: run directory missing: {LUMLEY_RUN}"
    )


# ---------------------------------------------------------------------------
# 8. VERIFY REQUIRED SCIENTIFIC OUTPUTS
# ---------------------------------------------------------------------------

required_outputs = (
    "run_manifest.json",
    "anisotropy_summary.json",
    "diagnostics.json",
    "metrics.csv",
    "uncertainty.csv",
    "methods.md",
)

missing_outputs = [
    name
    for name in required_outputs
    if not (LUMLEY_RUN / name).is_file()
]

if missing_outputs:
    raise RuntimeError(
        "LUMLEY_ABORT: missing required outputs: "
        + ", ".join(missing_outputs)
    )


lumley_manifest = read_json(
    LUMLEY_RUN / "run_manifest.json"
)

if lumley_manifest.get("status") != "COMPLETE":
    raise RuntimeError(
        "LUMLEY_ABORT: anisotropy run did not finalize COMPLETE."
    )

if lumley_manifest.get("git_commit") != LUMLEY_COMMIT:
    raise RuntimeError(
        "LUMLEY_ABORT: run-manifest commit mismatch.\n"
        f"Expected: {LUMLEY_COMMIT}\n"
        f"Observed: {lumley_manifest.get('git_commit')}"
    )


lumley_summary = read_json(
    LUMLEY_RUN / "anisotropy_summary.json"
)

lumley_diagnostics = read_json(
    LUMLEY_RUN / "diagnostics.json"
)

lumley_phases = lumley_summary.get("phases", {})


# ---------------------------------------------------------------------------
# 9. ENFORCE ONLY THE EXISTING GATE-A HANDOFF
#
# No desired anisotropy outcome is enforced.
# ---------------------------------------------------------------------------

if (
    lumley_phases.get("t018", {}).get("status")
    != "SKIPPED_GATE_A"
):
    raise RuntimeError(
        "LUMLEY_ABORT: t018 entered anisotropy analysis "
        "despite unresolved Gate A."
    )

for phase in ("t021", "t024"):
    status = lumley_phases.get(
        phase, {}
    ).get("status")

    if status != "CHARACTERIZED":
        raise RuntimeError(
            f"LUMLEY_ABORT: {phase} was not successfully "
            f"characterized: status={status!r}"
        )


# ---------------------------------------------------------------------------
# 10. VERIFY THE NEW PLOT-READY NODEWISE OUTPUTS
# ---------------------------------------------------------------------------

expected_field_set = set(EXPECTED_PLOT_FIELDS)

plot_data_records = {}

for phase in ("t021", "t024"):
    phase_diag = lumley_diagnostics[
        "phases"
    ][phase]

    plot_info = phase_diag.get("plot_data")

    if not isinstance(plot_info, dict):
        raise RuntimeError(
            f"LUMLEY_ABORT: missing plot-data metadata for {phase}."
        )

    relpath = plot_info.get("path")
    plot_path = LUMLEY_RUN / relpath

    if not plot_path.is_file():
        raise RuntimeError(
            f"LUMLEY_ABORT: plot-ready NPZ missing for "
            f"{phase}: {plot_path}"
        )

    with np.load(
        plot_path,
        allow_pickle=False,
    ) as npz:

        observed_fields = set(npz.files)

        if observed_fields != expected_field_set:
            raise RuntimeError(
                f"LUMLEY_ABORT: unexpected NPZ field set "
                f"for {phase}.\n"
                f"Expected: {sorted(expected_field_set)}\n"
                f"Observed: {sorted(observed_fields)}"
            )

        node_index = npz["node_index"]
        n = int(node_index.shape[0])

        expected_n = int(
            phase_diag["valid_nodes"]
        )

        if n != expected_n:
            raise RuntimeError(
                f"LUMLEY_ABORT: node count mismatch for {phase}: "
                f"{n} vs {expected_n}"
            )

        expected_shapes = {
            "coordinates": (n, 3),
            "k": (n,),
            "b": (n, 3, 3),
            "II": (n,),
            "III": (n,),
            "eigenvalues": (n, 3),
            "barycentric": (n, 3),
            "realizable": (n,),
        }

        for field, expected_shape in expected_shapes.items():
            observed_shape = npz[field].shape

            if observed_shape != expected_shape:
                raise RuntimeError(
                    f"LUMLEY_ABORT: {phase} field {field!r} "
                    f"has shape {observed_shape}; "
                    f"expected {expected_shape}."
                )

    plot_data_records[phase] = {
        "path": str(plot_path),
        "valid_nodes": int(
            phase_diag["valid_nodes"]
        ),
        "fields": list(EXPECTED_PLOT_FIELDS),
    }


# ---------------------------------------------------------------------------
# 11. VERIFY NPZ FILES ARE HASHED IN THE RUN MANIFEST
# ---------------------------------------------------------------------------

manifest_output_paths = {
    item["path"]
    for item in lumley_manifest.get("outputs", [])
}

for phase in ("t021", "t024"):
    relative_npz = str(
        Path(
            lumley_diagnostics[
                "phases"
            ][phase]["plot_data"]["path"]
        )
    )

    if relative_npz not in manifest_output_paths:
        raise RuntimeError(
            f"LUMLEY_ABORT: {relative_npz} is not recorded "
            "in the finalized run manifest."
        )


# ---------------------------------------------------------------------------
# 12. DISPLAY THE ANALYSIS RESULTS
# ---------------------------------------------------------------------------

lumley_metrics = pd.read_csv(
    LUMLEY_RUN / "metrics.csv"
)

lumley_uncertainty = pd.read_csv(
    LUMLEY_RUN / "uncertainty.csv"
)


print("\nLumley anisotropy analysis completed.")
print("Lumley run:", LUMLEY_RUN)
print("Lumley-stage commit:", lumley_manifest["git_commit"])

print("\nPhase summary:")
for phase in ("t018", "t021", "t024"):
    print(
        f"  {phase}: "
        f"{lumley_phases.get(phase)}"
    )

print("\nAnisotropy metrics:")
display(lumley_metrics)

print("\nRealization-bootstrap barycentric uncertainty:")
display(lumley_uncertainty)


print("\nPlot-ready nodewise data:")
for phase in ("t021", "t024"):
    print(
        f"  {phase}: "
        f"{plot_data_records[phase]['valid_nodes']} nodes"
    )
    print(
        "     ",
        plot_data_records[phase]["path"],
    )


print("\nAvailable nodewise fields:")
for field in EXPECTED_PLOT_FIELDS:
    print(" ", field)


print("\nExisting quick-look barycentric figures:")
for phase in ("t021", "t024"):
    figure_path = (
        LUMLEY_RUN
        / "figures"
        / f"{phase}_barycentric_map.png"
    )

    print(
        f"  {phase}: "
        f"{figure_path if figure_path.exists() else 'not generated'}"
    )


print("\nLater publication plotting can use:")
print("  Classical Lumley invariant map: II versus III")
print("  Barycentric realizability triangle: C1, C2, C3")
print("  Spatial anisotropy maps using saved native coordinates")
print("  Eigenvalue and k distributions")
print("  Realizability filtering")


# ---------------------------------------------------------------------------
# 13. NOTEBOOK-LEVEL EXECUTION POINTER
#
# Does not modify finalized analysis outputs.
# Does not promote anything to 05_results.
# ---------------------------------------------------------------------------

record_dir = (
    Path(WORKSPACE_ROOT)
    / "colab_runs"
)

record_dir.mkdir(
    parents=True,
    exist_ok=True,
)

record_path = record_dir / (
    "lumley_"
    + datetime.now(timezone.utc).strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".json"
)

record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "upstream_scientific_commit": UPSTREAM_COMMIT,
    "lumley_stage_commit": LUMLEY_COMMIT,

    "dataset_root": str(
        Path(DATASET_ROOT).resolve()
    ),

    "decomposition_run": str(
        Path(DECOMPOSITION_RUN).resolve()
    ),

    "regime_run": str(
        Path(REGIME_RUN).resolve()
    ),

    "kolmogorov_run": str(
        Path(KO_RUN).resolve()
    ),

    "taylor_run": str(
        Path(TAYLOR_RUN).resolve()
    ),

    "lumley_run": str(
        LUMLEY_RUN
    ),

    "formal_gate_a": {
        phase: {
            "decision":
                gate_phases[phase].get(
                    "decision"
                ),
            "turbulence_established":
                gate_phases[phase].get(
                    "turbulence_established"
                ),
        }
        for phase in (
            "t018",
            "t021",
            "t024",
        )
    },

    "ko_status": {
        phase:
            ko_phases.get(
                phase, {}
            ).get("status")
        for phase in (
            "t018",
            "t021",
            "t024",
        )
    },

    "taylor_status": {
        phase:
            taylor_phases.get(
                phase, {}
            ).get("status")
        for phase in (
            "t018",
            "t021",
            "t024",
        )
    },

    "lumley_summary": lumley_phases,

    "plot_ready_nodewise_data": plot_data_records,

    "canonical_root_05_results_written_by_notebook": False,
}

record_path.write_text(
    json.dumps(
        record,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print(
    "\nLumley execution record:",
    record_path,
)

print(
    "Canonical root 05_results written by this cell: False"
)

Locked upstream sequence verified:
  t018 -> Gate A unresolved; excluded
  t021 -> turbulence established; KO unresolved; Taylor not established
  t024 -> turbulence established; KO unresolved; Taylor not established

Installing exact Lumley persistence commit:
  c74cdd8ca072f426a118b30df9ea98d8ae7372a2
Lumley-stage package provenance verified:
  commit : c74cdd8ca072f426a118b30df9ea98d8ae7372a2
  version: 0.2.0
  module : /content/aorticturb-lumley-site-c74cdd8ca072/aorticturb/__init__.py

Starting Lumley/Reynolds-stress anisotropy analysis...

__AORTICTURB_LUMLEY_RESULT__={"result": {"run_dir": "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/04_lumley_anisotropy/run_20261024_113954_081943", "phases": {"t018": {"status": "SKIPPED_GATE_A"}, "t021": {"status": "CHARACTERIZED", "realizability_fraction": 1.0, "median_barycentric": [0.3222693442402588, 0.2934304490476139, 0.34709583093794816], "plot_data": "processed/t021_anisotropy.npz"}, "t024": {"statu

,phase,status,valid_nodes,realizability_fraction,II_median,III_median,C1_median,C2_median,C3_median
0,t018,SKIPPED_GATE_A,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,t021,CHARACTERIZED,386967.0,1.0,-0.062681,0.002914,0.322269,0.293430,0.347096
2,t024,CHARACTERIZED,386983.0,1.0,-0.062076,0.002734,0.316640,0.298949,0.344674



Realization-bootstrap barycentric uncertainty:


,phase,quantity,lower,median,upper,bootstrap_axis
0,t021,C1_one_component,0.168515,0.186100,0.200311,realizations
1,t021,C2_two_component,0.017670,0.039231,0.061158,realizations
2,t021,C3_three_component,0.751788,0.775807,0.794845,realizations
3,t024,C1_one_component,0.166869,0.180864,0.194775,realizations
4,t024,C2_two_component,0.005475,0.024322,0.043911,realizations
5,t024,C3_three_component,0.774637,0.794504,0.815640,realizations



Plot-ready nodewise data:
  t021: 386967 nodes
      /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/04_lumley_anisotropy/run_20261024_113954_081943/processed/t021_anisotropy.npz
  t024: 386983 nodes
      /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/04_lumley_anisotropy/run_20261024_113954_081943/processed/t024_anisotropy.npz

Available nodewise fields:
  node_index
  coordinates
  k
  b
  II
  III
  eigenvalues
  barycentric
  realizable

Existing quick-look barycentric figures:
  t021: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/04_lumley_anisotropy/run_20261024_113954_081943/figures/t021_barycentric_map.png
  t024: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/04_lumley_anisotropy/run_20261024_113954_081943/figures/t024_barycentric_map.png

Later publication plotting can use:
  Classical Lumley invariant map: II versus III
  Barycentric r